# Open-RAG-Eval

> 中文学习版导读｜原始案例：`open-rag-eval-example.ipynb`  
> 所属阶段：第八阶段：评估与回归

## 本节目标

使用开放评估工具检查相关性、引用和幻觉。

## 阅读重点

把自动指标与人工抽样复核结合使用。

## 运行约定

- 从项目根目录启动 JupyterLab。
- 模型和服务地址统一读取 `config/models.toml`。
- API Key 等敏感信息只写入本地 `.env`。
- 本 Notebook 保留上游实现用于技术核对；新增中文导读负责说明学习顺序、配置方式和实验重点。
- 运行前阅读同目录的 `open-rag-eval-example.zh-CN.md`。


In [ ]:
# 统一配置入口：模型名和服务地址来自 config/models.toml，密钥来自 .env
from pathlib import Path
import os
import sys

_current = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (_current, *_current.parents)
    if (candidate / "pyproject.toml").exists()
)
_src = str(PROJECT_ROOT / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from rag_techniques_zh.config import apply_runtime_environment
settings = apply_runtime_environment()
CHAT_MODEL = settings.openai.chat_model
EMBEDDING_MODEL = settings.openai.embedding_model
EVALUATION_MODEL = settings.openai.evaluation_model
OPENAI_API_KEY = settings.openai.api_key
OPENAI_BASE_URL = settings.openai.base_url
CONTEXTUAL_BASE_URL = settings.contextual.base_url
COHERE_CHAT_MODEL = settings.cohere.chat_model
COHERE_EMBEDDING_MODEL = settings.cohere.embedding_model
COHERE_RERANK_MODEL = settings.cohere.rerank_model
GOOGLE_CHAT_MODEL = settings.google.chat_model
GROQ_FAST_MODEL = settings.groq.fast_model
GROQ_LARGE_MODEL = settings.groq.large_model
OLLAMA_EMBEDDING_MODEL = settings.ollama.embedding_model
SENTENCE_TRANSFORMER_MODEL = settings.local_models.sentence_transformer_model
CROSS_ENCODER_MODEL = settings.local_models.cross_encoder_model
CONTEXTUAL_RERANK_MODEL = settings.contextual.rerank_model
if settings.cohere.api_key:
    os.environ.setdefault("CO_API_KEY", settings.cohere.api_key)

print("当前模型配置：", {
    "chat": CHAT_MODEL,
    "embedding": EMBEDDING_MODEL,
    "evaluation": EVALUATION_MODEL,
    "base_url": OPENAI_BASE_URL,
})


# Evaluating RAG with Open-RAG-Eval

## Overview

[Open-RAG-Eval](https://github.com/vectara/open-rag-eval) is an open-source framework that lets teams evaluate RAG systems without needing predefined (golden) answers, making it faster and easier to compare solutions or configurations. With automated, research-backed metrics like UMBRELA and Hallucination, it brings transparency and rigor to RAG performance testing at any scale.

This notebook provides a simple example of "how to use" Open-RAG-Eval. 
You can read more about the overall vision and metrics in this [blog post](https://www.vectara.com/blog/towards-a-gold-standard-for-rag-evaluation)

Specifically, we run the evaluation framework on the [fiqa](https://sites.google.com/view/fiqa/home) dataset from the [BEIR benchmark](https://library.toponeai.link/dataset/beir), which contains question and answer paris for the financial domain. 

In a typical RAG evaluation flow - one would need to create a RAG pipeline, run the queries againat that RAG pipeline adn collect the chunks and responses. Since this type of flow doesn't fit well in a notebook, we already pre-indexed the dataset in Vectara and ran a subset of queries (from the same fiqa dataset) against the indexed data, and stored the outputs in the [fiqa_output.csv](https://raw.githubusercontent.com/vectara/example-notebooks/main/data/fiqa_output.csv) file, which contains the queries, retrieved results and LLM generated answers to those queries.

If you have a Vectara account, you can set Open-RAG-Eval with the Vectara connector and a set of queries to automatically create this file by runnning each query through your Vectara RAG.


## RAG Evaluation outputs

To get started, let's look at the `fiqa_output.csv` file:

In [ ]:
import pandas as pd
import os

from dotenv import load_dotenv

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from open_rag_eval.evaluators import TRECEvaluator
from open_rag_eval.models.llm_judges import OpenAIModel
from open_rag_eval.connectors import CSVConnector
from open_rag_eval.data_classes import eval_scores

import pprint
pp = pprint.PrettyPrinter(indent=4, width=80)

df = pd.read_csv('https://raw.githubusercontent.com/vectara/example-notebooks/main/data/fiqa_output.csv').fillna('')
df = df.head(50) # Limiting to first 50 rows for faster run time
df.head(10)

`fiqa_output.csv` comes in the CSV format that Open-RAG-Eval expects.

* Each query has a unique `query_id`, with multiple rows corresponding to each query. This is because we use one row per retrieved chunk, and in this case we retrieved 5 passages per query, and you can see that for each passage we have a separate passage_id and passage text.
* The `generated_answer` column includes the answer generated by the LLM to the query grounded on the passages.

## Run an Open-RAG-Eval Evaluation

To run a full evaluation, we need to 
1. Specify the evaluator: for this example we're using the TREC RAG Evaluator. This Evaluator needs an LLM as the judge model which will be scoring responses, in this case we will use GPT-4o-mini.
2. Creatae a `CSVConnector` to read the data and simply run the evaluator
3. Run the evaluation

In [ ]:
# Make sure your OPENAI_API_KEY is in a .env file in the same directory as this notebook.
load_dotenv()

judge_model = OpenAIModel(model_name=EVALUATION_MODEL, api_key=os.getenv("OPENAI_API_KEY"))
evaluator = TRECEvaluator(model=judge_model)

rag_results = CSVConnector('https://raw.githubusercontent.com/vectara/example-notebooks/main/data/fiqa_output.csv').fetch_data()
scored_results = evaluator.evaluate_batch(rag_results)

Note: you can ignore the 'Token indices sequence length is longer than the specified maximum sequence length for this model (1782 > 512). It's not important.

## Review Evaluation Results

The evaluator uses multiple metrics.

Internally, each `metric` class scores either the retrieval results or the generated response and adds its own scores to the `scored_results` object under a unique key, which contain both the original query, retrieved passages and the scored metrics along with any intermediate metric output for easy debugging and in-depth analysis.

Let's look at the metrics in more depth, starting with the `UMBRELA` metric:

### UMBRELA Metric

The `UMBRELA` metric assigns a value to each retrieved passage as follows:

* NO_RELEVANCE = "0"
* RELATED = "1"
* PARTIAL_ANSWER = "2"
* EXACT_ANSWER = "3"

Let's pick a query to show how the `UMBRELA` metrics look:

In [ ]:
for scored_result in scored_results:
    if scored_result.rag_result.retrieval_result.query == "accredited investor definition":
        break

scores = scored_results[0].scores

 # Our query is:
pp.pprint(f"Query: {scored_result.rag_result.retrieval_result.query}")
pp.pprint("------")

num_results_to_show = 5
for key, passage in scored_result.rag_result.retrieval_result.retrieved_passages.items():
    pp.pprint(f"Passage: {passage}")
    pp.pprint(f"Score: {scores.retrieval_score.scores['umbrela_scores'][key]}")
    pp.pprint("------")
    num_results_to_show -= 1
    if num_results_to_show == 0:
        break

### AutoNuggetizer Metric

Another interesting metric to look at is the `autoNuggetizer` metric

We can examine which nuggets of information the metric thought a good answer should have, based on the passages (the so-called 'auto nuggets'). 
Each nugget is deemed 'vital' or 'okay' to be included in the answer, and then the generated answer is compared to each nugget to see if it "contains", "partially contains" or "does not contain" the nugget (called "support", "partial_support" and "not_support")

In [ ]:
autonuggetizer_scores = scores.generation_score.scores['autonugget_scores']

# Let's look at the generated answer first (this needs a bit of construction as we decompose and store it as text + citations).
recreated_answer = []
for part in scored_result.rag_result.generation_result.generated_answer:
    recreated_answer.append(part.text)
    for citation in part.citations:
        recreated_answer.append(citation)
generated_answer =" ".join(recreated_answer)
pp.pprint(f"Generated Answer: {generated_answer}")

for nugget, importance, support in zip(autonuggetizer_scores['nuggets'], autonuggetizer_scores['labels'], autonuggetizer_scores['assignments']):    
    pp.pprint(f"Nugget: {nugget}, Importance: {importance}, Support: {support}")

### Hallucination Metric

This metric indicates whether the generated answer that you see in the previous section is hallucinated or not based on the retrieved passages that you can see in the UMBRELA section. 

The score ranges from [0, 1] with a higher score indicating a higher consistency, we usually threshold the score at 0.5 (so scores >= 0.5 is considered 'factually consistent' and scores < 0.5 are considered hallucinated).


In [ ]:
print("Hallucination Score:", scores.generation_score.scores['hallucination_scores'])


### Citation Metric
The citation metric determines if the citations produced by the generative model are accurate or not. If for example the LLM says a sentence came from retrieved passage [1], does that passage actually support the sentence or not. You can view a per citation view of this or an aggregate F1 score.


In [ ]:
print("Citation Scores:")
scores.generation_score.scores['citation_scores']

## Plotting results

Now that we have the `scored_results`, let's save them into a file, and then use the `plot_metrics` function to create a summary plot of the evaluation results. 

If you have multiple results files you can pass them to the `plot_metrics` function as a list and all the results will be plotted on the same graph object. Here we only have one.

In [ ]:

# Save the evaluation results to a CSV file
eval_scores.to_csv(scored_results, file_path="evaluation_results.csv")

# Plot the metrics
TRECEvaluator.plot_metrics(csv_files=["evaluation_results.csv"], output_file="metrics_summary.png")


In [ ]:
plt.figure(figsize=(10, 8))

img = mpimg.imread('metrics_summary.png')
plt.imshow(img)
plt.axis('off')
plt.show()